# 2. HDBSCAN-пайплайн по подготовленной raw-выборке

Этот ноутбук **не читает raw-директорию напрямую**. Он берет уже подготовленные sample-файлы из директории:

```text
/content/drive/MyDrive/MLops_project/project/data/hdbscan_raw_sample_200_files_1000_each/
```

Пайплайн:

1. Читает sample parquet-файлы.
2. Очищает текстовую колонку.
3. Строит эмбеддинги через `SentenceTransformer`.
4. Сохраняет эмбеддинги частями, чтобы не терять прогресс.
5. Делает UMAP.
6. Запускает HDBSCAN.
7. Сохраняет итоговую таблицу с `cluster_id`.

In [1]:
# Если запускаешь в Colab, сначала подключи Google Drive
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Установка зависимостей
!pip -q install sentence-transformers umap-learn hdbscan pyarrow fastparquet tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 31.1 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import json
import time
import threading

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

import umap
import hdbscan

## Настройки

In [4]:
BASE_DIR = Path('/content/drive/MyDrive/MLops_project/project')
DATA_DIR = BASE_DIR / 'data'

SAMPLE_DIR = DATA_DIR / 'hdbscan_raw_sample_200_files_1000_each'
RESULT_DIR = DATA_DIR / 'hdbscan_results_raw_sample_200k'
EMBEDDINGS_DIR = RESULT_DIR / 'embeddings_parts'

RESULT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# Если None — читаем все sample-файлы из SAMPLE_DIR.
# Для быстрого теста можно поставить, например, 10, тогда будет 10_000 отзывов.
MAX_SAMPLE_FILES = None

TEXT_COL_CANDIDATES = [
    'text', 'review', 'comment', 'feedback', 'content',
    'description', 'message', 'review_text', 'body'
]

MIN_TEXT_LEN = 5
RANDOM_SEED = 42

EMBEDDING_MODEL_NAME = 'intfloat/multilingual-e5-base'
E5_PREFIX = 'passage: '
EMBEDDING_BATCH_SIZE = 128

# UMAP
UMAP_N_COMPONENTS = 15
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.0
UMAP_METRIC = 'cosine'

# HDBSCAN
MIN_CLUSTER_SIZE = 50
MIN_SAMPLES = 10
HDBSCAN_METRIC = 'euclidean'

print('SAMPLE_DIR:', SAMPLE_DIR)
print('RESULT_DIR:', RESULT_DIR)
print('EMBEDDINGS_DIR:', EMBEDDINGS_DIR)
print('CUDA available:', torch.cuda.is_available())

SAMPLE_DIR: /content/drive/MyDrive/MLops_project/project/data/hdbscan_raw_sample_200_files_1000_each
RESULT_DIR: /content/drive/MyDrive/MLops_project/project/data/hdbscan_results_raw_sample_200k
EMBEDDINGS_DIR: /content/drive/MyDrive/MLops_project/project/data/hdbscan_results_raw_sample_200k/embeddings_parts
CUDA available: True


## Загрузка sample-файлов

In [5]:
if not SAMPLE_DIR.exists():
    raise FileNotFoundError(f'Не найдена директория с выборкой: {SAMPLE_DIR}. Сначала запусти первый ноутбук.')

sample_files = sorted(SAMPLE_DIR.glob('sample_part_*.parquet'))
if MAX_SAMPLE_FILES is not None:
    sample_files = sample_files[:MAX_SAMPLE_FILES]

print(f'Будет прочитано sample-файлов: {len(sample_files)}')

if len(sample_files) == 0:
    raise FileNotFoundError(f'В {SAMPLE_DIR} нет sample_part_*.parquet')

parts = []
for p in tqdm(sample_files, desc='Loading sample parquet files'):
    part = pd.read_parquet(p)
    part['sample_file'] = p.name
    parts.append(part)

df = pd.concat(parts, ignore_index=True)
print('Размер df:', df.shape)
display(df.head(3))

Будет прочитано sample-файлов: 190


Loading sample parquet files:   0%|          | 0/190 [00:00<?, ?it/s]

Размер df: (190000, 10)


,nmId,productValuation,color,text,answer,source_row_idx,source_file,source_text_col,sample_part_id,sample_file
0,0,5,темно-серый,качество хорошее. цвет темно серый. Очень прог...,,75721,raw_part_00000.parquet,text,0,sample_part_00000.parquet
1,0,5,черный,Классная лампа. Не хлипкая. Очень удобная. Бра...,Добрый день! благодарим за выбор нашей продукц...,80184,raw_part_00000.parquet,text,0,sample_part_00000.parquet
2,0,4,черный,На узкую ногу,Здравствуйте! Спасибо за отзыв. Мы очень сожал...,19864,raw_part_00000.parquet,text,0,sample_part_00000.parquet


## Поиск и очистка текстовой колонки

In [6]:
def detect_text_col(columns, candidates=TEXT_COL_CANDIDATES):
    columns = list(columns)
    for col in candidates:
        if col in columns:
            return col
    return None

TEXT_COL = detect_text_col(df.columns)
if TEXT_COL is None:
    raise ValueError(f'Не удалось найти текстовую колонку. Колонки: {list(df.columns)}')

print('TEXT_COL:', TEXT_COL)

work_df = df.copy()
work_df[TEXT_COL] = (
    work_df[TEXT_COL]
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

before = len(work_df)
work_df = work_df[work_df[TEXT_COL].str.len() >= MIN_TEXT_LEN].copy()
work_df = work_df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

after = len(work_df)
print(f'Строк до очистки: {before}')
print(f'Строк после очистки и удаления дублей: {after}')
display(work_df[[TEXT_COL]].head(5))

TEXT_COL: text
Строк до очистки: 190000
Строк после очистки и удаления дублей: 178132


,text
0,качество хорошее. цвет темно серый. Очень прог...
1,Классная лампа. Не хлипкая. Очень удобная. Бра...
2,На узкую ногу
3,"Штаны хорошие, обычные брюки цена в 1300 в сам..."
4,"Думали порвутся сразу, но как подарок на нг со..."


## Построение эмбеддингов с сохранением по батчам

Эмбеддинги сохраняются в `embeddings_parts`. Если Colab упадет, уже посчитанные батчи не потеряются.

In [7]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

texts = work_df[TEXT_COL].astype(str).tolist()
texts_for_embedding = [E5_PREFIX + t for t in texts]

num_batches = int(np.ceil(len(texts_for_embedding) / EMBEDDING_BATCH_SIZE))
print('Всего текстов:', len(texts_for_embedding))
print('Всего embedding-батчей:', num_batches)

Device: cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Всего текстов: 178132
Всего embedding-батчей: 1392


In [8]:
embedding_meta = {
    'embedding_model_name': EMBEDDING_MODEL_NAME,
    'e5_prefix': E5_PREFIX,
    'embedding_batch_size': EMBEDDING_BATCH_SIZE,
    'num_texts': len(texts_for_embedding),
    'text_col': TEXT_COL,
}
(RESULT_DIR / 'embedding_meta.json').write_text(json.dumps(embedding_meta, ensure_ascii=False, indent=2), encoding='utf-8')

for batch_idx in tqdm(range(num_batches), desc='Embedding batches'):
    part_path = EMBEDDINGS_DIR / f'emb_part_{batch_idx:06d}.npy'

    if part_path.exists():
        continue

    start = batch_idx * EMBEDDING_BATCH_SIZE
    end = min(start + EMBEDDING_BATCH_SIZE, len(texts_for_embedding))
    batch = texts_for_embedding[start:end]

    emb = model.encode(
        batch,
        batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=False,
    ).astype('float32')

    tmp_path = EMBEDDINGS_DIR / f'.emb_part_{batch_idx:06d}.tmp.npy'
    np.save(tmp_path, emb)
    tmp_path.rename(part_path)

print('Готово. Эмбеддинги по батчам сохранены.')

Embedding batches:   0%|          | 0/1392 [00:00<?, ?it/s]

Готово. Эмбеддинги по батчам сохранены.


## Сборка полной матрицы эмбеддингов

In [9]:
part_paths = sorted(EMBEDDINGS_DIR.glob('emb_part_*.npy'))
print('Найдено embedding parts:', len(part_paths))

if len(part_paths) != num_batches:
    raise RuntimeError(f'Ожидалось {num_batches} embedding parts, найдено {len(part_paths)}')

emb_parts = []
for p in tqdm(part_paths, desc='Loading embedding parts'):
    emb_parts.append(np.load(p))

embeddings = np.vstack(emb_parts).astype('float32')
embeddings = normalize(embeddings).astype('float32')

print('Embeddings shape:', embeddings.shape)

full_embeddings_path = RESULT_DIR / 'embeddings_full_normalized.npy'
np.save(full_embeddings_path, embeddings)
print('Сохранена полная матрица эмбеддингов:', full_embeddings_path)

Найдено embedding parts: 1392


Loading embedding parts:   0%|          | 0/1392 [00:00<?, ?it/s]

Embeddings shape: (178132, 768)
Сохранена полная матрица эмбеддингов: /content/drive/MyDrive/MLops_project/project/data/hdbscan_results_raw_sample_200k/embeddings_full_normalized.npy


## UMAP

UMAP снижает размерность эмбеддингов перед HDBSCAN.

In [10]:
reducer = umap.UMAP(
    n_components=UMAP_N_COMPONENTS,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=RANDOM_SEED,
    verbose=True,
)

X_umap = reducer.fit_transform(embeddings).astype('float32')
print('X_umap shape:', X_umap.shape)

umap_path = RESULT_DIR / 'umap_features.npy'
np.save(umap_path, X_umap)
print('UMAP features сохранены:', umap_path)

UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.0, n_components=15, n_jobs=1, n_neighbors=30, random_state=42, verbose=True)


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Fri Jun  5 17:50:14 2026 Construct fuzzy simplicial set
Fri Jun  5 17:50:14 2026 Finding Nearest Neighbors
Fri Jun  5 17:50:14 2026 Building RP forest with 26 trees
Fri Jun  5 17:50:40 2026 NN descent for 17 iterations
	 1  /  17
	 2  /  17
	 3  /  17
	 4  /  17
	 5  /  17
	Stopping threshold met -- exiting after 5 iterations
Fri Jun  5 17:52:53 2026 Finished Nearest Neighbor Search
Fri Jun  5 17:53:02 2026 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Fri Jun  5 18:00:25 2026 Finished embedding
X_umap shape: (178132, 15)
UMAP features сохранены: /content/drive/MyDrive/MLops_project/project/data/hdbscan_results_raw_sample_200k/umap_features.npy


## HDBSCAN

У HDBSCAN нет честного внутреннего progress bar. Поэтому ниже используется таймер, который показывает, что ячейка еще выполняется.

In [11]:
def run_with_timer(fn, desc='Running'):
    done = {'value': False}

    def timer_loop():
        start = time.time()
        with tqdm(total=0, bar_format='{desc} | elapsed: {elapsed}', desc=desc) as pbar:
            while not done['value']:
                time.sleep(1)
                pbar.update(0)

    thread = threading.Thread(target=timer_loop, daemon=True)
    thread.start()
    try:
        return fn()
    finally:
        done['value'] = True
        thread.join(timeout=2)


def fit_hdbscan():
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric=HDBSCAN_METRIC,
        prediction_data=True,
    )
    labels = clusterer.fit_predict(X_umap)
    return clusterer, labels

clusterer, labels = run_with_timer(fit_hdbscan, desc='HDBSCAN')

print('Готово')
print('Всего кластеров без шума:', len(set(labels)) - (1 if -1 in labels else 0))
print('Доля шума label=-1:', float(np.mean(labels == -1)))

HDBSCAN | elapsed: 00:00

Готово
Всего кластеров без шума: 297
Доля шума label=-1: 0.5083926526396155


## Сохранение результата

In [12]:
result_df = work_df.copy()
result_df['cluster_id'] = labels

# Вероятность принадлежности к кластеру. Для шума обычно низкая.
if hasattr(clusterer, 'probabilities_'):
    result_df['cluster_probability'] = clusterer.probabilities_

result_parquet_path = RESULT_DIR / 'hdbscan_clusters.parquet'
result_csv_path = RESULT_DIR / 'hdbscan_clusters.csv'

result_df.to_parquet(result_parquet_path, index=False)
result_df.to_csv(result_csv_path, index=False)

print('Сохранено:')
print(result_parquet_path)
print(result_csv_path)

Сохранено:
/content/drive/MyDrive/MLops_project/project/data/hdbscan_results_raw_sample_200k/hdbscan_clusters.parquet
/content/drive/MyDrive/MLops_project/project/data/hdbscan_results_raw_sample_200k/hdbscan_clusters.csv


## Сводка по кластерам

In [13]:
cluster_summary = (
    result_df
    .groupby('cluster_id')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

cluster_summary['share'] = cluster_summary['count'] / len(result_df)
summary_path = RESULT_DIR / 'cluster_summary.csv'
cluster_summary.to_csv(summary_path, index=False)

print('Cluster summary:', summary_path)
display(cluster_summary.head(30))

Cluster summary: /content/drive/MyDrive/MLops_project/project/data/hdbscan_results_raw_sample_200k/cluster_summary.csv


,cluster_id,count,share
0,-1,90561,0.508393
276,275,3966,0.022264
218,217,2821,0.015837
295,294,2776,0.015584
254,253,2442,0.013709
55,54,2212,0.012418
9,8,1789,0.010043
259,258,1765,0.009908
229,228,1708,0.009588
278,277,1538,0.008634


## Примеры отзывов из кластеров

In [14]:
def show_cluster_examples(cluster_id, n=10):
    subset = result_df[result_df['cluster_id'] == cluster_id]
    print(f'cluster_id={cluster_id}, size={len(subset)}')
    cols = [TEXT_COL]
    if 'cluster_probability' in result_df.columns:
        cols.append('cluster_probability')
    display(subset[cols].sample(min(n, len(subset)), random_state=RANDOM_SEED))

# Покажем несколько самых больших не-шумовых кластеров
big_clusters = cluster_summary[cluster_summary['cluster_id'] != -1]['cluster_id'].head(10).tolist()
for cid in big_clusters:
    show_cluster_examples(cid, n=5)

cluster_id=275, size=3966


,text,cluster_probability
8308,"Пришел целый, коробка не мятая. Спасибо большо...",0.9503
82254,Мягкая игрушка отличного качества! Пришла тако...,1.0000
165245,"Удобные, ребенку на физру самое то",1.0000
38171,"Отличная игрушка, дочке понравилась.",1.0000
17613,"Игрушка пришла, отличного качества, ребёнку оч...",1.0000


cluster_id=217, size=2821


,text,cluster_probability
76133,Хорошая юбка.Но ошиблась с размером.Перезакажу.,1.0
150934,"Вполне хорошая,но пена юыстро оседает",1.0
54316,"Хороший держатель, только коротенький",1.0
77044,"Очень хорошие сапожки, но маломерят как миниму...",1.0
147940,"Большие оказались, но все равно забрала",1.0


cluster_id=294, size=2776


...

,text,cluster_probability
74700,Это бомба 🔥,1.000000
144226,Всё классно 👍,1.000000
146342,Стул 💥💥💥💥💥💥💥💥,1.000000
18654,Крутой! Очень стильный!,0.800129
64307,Очень даже!👍,1.000000


cluster_id=277, size=1538


,text,cluster_probability
76142,Доставка четко и в срок. Спасибо.,0.984541
14855,"Все подошло, доставка быстрая спасибо!!! 👍👍👍",0.777147
174653,Все пришло в отличном состоянии. Спасибо!,0.908360
93543,"приехало в хорошем состоянии, очень круто и уд...",0.804528
162638,Спасибо за быструю доставку. Упаковка целая. К...,0.786670


cluster_id=234, size=1456


,text,cluster_probability
75788,"Классный, рекомендую",1.00000
158196,"классные грили, рекомендую.",1.00000
67087,Очень круто всем советою,1.00000
136642,"Хорошие ушки, советую.",1.00000
134879,"Аптечка хорошая, рекомендую",0.99211


## Быстрый подбор параметров HDBSCAN

Этот блок можно запускать после основного результата, чтобы посмотреть, как меняется число кластеров и доля шума.

In [15]:
param_results = []

for min_cluster_size in [30, 50, 100, 200]:
    for min_samples in [5, 10, 20]:
        clusterer_tmp = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric=HDBSCAN_METRIC,
        )
        labels_tmp = clusterer_tmp.fit_predict(X_umap)
        n_clusters = len(set(labels_tmp)) - (1 if -1 in labels_tmp else 0)
        noise_share = float(np.mean(labels_tmp == -1))
        param_results.append({
            'min_cluster_size': min_cluster_size,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'noise_share': noise_share,
        })

param_df = pd.DataFrame(param_results)
display(param_df.sort_values(['noise_share', 'n_clusters']))
param_df.to_csv(RESULT_DIR / 'hdbscan_param_search.csv', index=False)

,min_cluster_size,min_samples,n_clusters,noise_share
11,200,20,103,0.468928
10,200,10,117,0.497609
1,30,10,413,0.505490
3,50,5,322,0.506815
0,30,5,456,0.507219
4,50,10,297,0.508393
8,100,20,190,0.512261
9,200,5,123,0.514220
2,30,20,362,0.518464
5,50,20,276,0.519261
